In [9]:
# 表格1: Performance Evaluation of LLM-Empowered Multi-Task Prediction with Different Sizes of Training Datasets.
import os
import pandas as pd

# 要统计的指标
target_metrics = ['task1_MAE', 'task2_average_accuracy', 'task3_Cosine_Similarity']
results = {metric: [] for metric in target_metrics}
# 指定根目录路径
root_path = "/data/LLM_indoor/LLaMA-Factory-main/LLM-train/Multitask/results/Dataset3_12k_RArevised-20251229-1213PM"
# 计数器，只处理前12个Dataset3开头的文件夹
dataset3_count = 0
# 遍历根目录下的所有文件夹，按字母顺序排序
sorted_folders = sorted(os.listdir(root_path))
for folder in sorted_folders:
    # 只处理Dataset3开头的文件夹
    if folder.startswith("Dataset3"):
        dataset3_count += 1
        
        # 只处理前12个Dataset3文件夹
        if dataset3_count > 12:
            break
            
        folder_path = os.path.join(root_path, folder)
        if os.path.isdir(folder_path):
            csv_path = os.path.join(folder_path, 'final_average_results.csv')
            if os.path.exists(csv_path):
                try:
                    df = pd.read_csv(csv_path)
                    print(f"Processing: {folder}")
                    for metric in target_metrics:
                        if metric in df['Metric'].values:
                            value = df.loc[df['Metric'] == metric, 'Average_Value'].iloc[0]
                            results[metric].append(value)
                except Exception as e:
                    print(f"Error reading {csv_path}: {e}")

# 计算并打印平均值
print(f"\n处理路径: {root_path}")
print("=" * 50)
print("指标平均值 (只处理前12个Dataset3文件):")
for metric, values in results.items():
    if values:
        avg_value = sum(values) / len(values)
        print(f"{metric}: {avg_value:.6f} (基于{len(values)}个文件)")
    else:
        print(f"{metric}: 未找到数据")

Processing: Dataset3_room1-1_sharegpt_5k_RArevised
Processing: Dataset3_room1-2_sharegpt_5k_RArevised
Processing: Dataset3_room2-1_sharegpt_5k_RArevised
Processing: Dataset3_room2-2_sharegpt_5k_RArevised
Processing: Dataset3_room3-1_sharegpt_5k_RArevised
Processing: Dataset3_room3-2_sharegpt_5k_RArevised
Processing: Dataset3_room4-1_sharegpt_5k_RArevised
Processing: Dataset3_room4-2_sharegpt_5k_RArevised
Processing: Dataset3_room5-1_sharegpt_5k_RArevised
Processing: Dataset3_room5-2_sharegpt_5k_RArevised
Processing: Dataset3_room6-1_sharegpt_5k_RArevised
Processing: Dataset3_room6-2_sharegpt_5k_RArevised

处理路径: /data/LLM_indoor/LLaMA-Factory-main/LLM-train/Multitask/results/Dataset3_12k_RArevised-20251229-1213PM
指标平均值 (只处理前12个Dataset3文件):
task1_MAE: 1.587980 (基于12个文件)
task2_average_accuracy: 0.646872 (基于12个文件)
task3_Cosine_Similarity: 0.666536 (基于12个文件)


In [5]:
# 图4 泛化性测试
import csv
from collections import defaultdict

def calculate_avg_performance_gap(file_path):
    ue_gap_dict = defaultdict(list)
    
    with open(file_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            ue_num = int(row['UE_num'])
            performance_gap = float(row['Performance_Gap'])
            ue_gap_dict[ue_num].append(performance_gap)
    
    ue_list = sorted(ue_gap_dict.keys())
    gap_list = [round(sum(ue_gap_dict[ue]) / len(ue_gap_dict[ue]), 3) for ue in ue_list]
    
    return ue_list, gap_list

data_path = '/data/LLM_indoor/LLaMA-Factory-main/LLM-train/Multitask/results/Dataset3_10k_RArevised_8B_lora-20260103-1145PM/Test5_Room5_RArevised/result_throuput.csv'
ue_list, gap_list = calculate_avg_performance_gap(data_path)
print(ue_list)
print(gap_list)

[5, 10, 15, 20, 25, 30]
[0.171, 0.004, 0.034, -0.067, -0.03, 0.035]


In [8]:
# 消融实验1: Proposed LLM vs LLM w/o Adapter
# 消融实验2: Proposed LLM vs LLM w/o Meta-Adapter
# 消融实验3: Proposed LLM vs MLP
from utils_new import mae, cosine_similarity, accuracy
import numpy as np
import re

# Room 1-1
SINR1 = '[[55.91 26.5 23.98 33.86 30.25] [54.73 28.79 21.25 35.05 25.3] [48.61 34.15 22.83 24.37 16.42] [47.16 14.66 21.31 24.23 34.17] [55.69 28.7 21.65 35.05 25.85] [54.49 29.77 21.51 34.65 24.59] [49.1 31.73 21.62 30.41 20.91] [54.31 34.41 29.67 25.93 22.75] [47.11 20.47 14.38 33.88 24.79] [51.36 34.03 22.58 29.42 19.8] [55.17 34.19 28.82 28.14 24.1] [49.12 13.86 20.89 23.67 33.69] [61.68 32.54 30.86 27.67 26.36] [53.86 35.04 24.25 28.48 20.13] [41.53 13.19 20.58 23.2 33.21]]'
SINR1_ture = [[10.33, 8.54, 5.23, 7.65, 9.18],
 [9.45, 7.18, 4.59, 6.92, 8.35],
 [8.63, 6.23, 4.19, 6.41, 7.62],
 [9.81, 7.54, 5.11, 7.33, 8.95],
 [8.09, 5.69, 3.98, 6.21, 7.42],
 [7.38, 5.18, 3.58, 5.91, 6.92],
 [6.69, 4.69, 3.23, 5.64, 6.45],
 [8.52, 6.29, 4.55, 6.87, 8.19],
 [7.01, 4.83, 3.46, 5.99, 7.11],
 [6.35, 4.39, 3.19, 5.63, 6.74],
 [7.87, 5.63, 4.25, 6.67, 8.09],
 [9.24, 7.01, 5.41, 7.13, 8.65],
 [8.51, 6.34, 4.67, 6.99, 8.21],
 [7.79, 5.69, 4.29, 6.53, 7.85],
 [6.09, 4.29, 3.51, 5.73, 6.94]]
SINR2 = '[[52.41 26.22 22.85 34.46 29.49] [52.58 29.36 21.08 34.81 24.51] [29.88 33.0 22.46 22.56 16.01] [46.24 14.6 20.53 25.0 33.98] [53.15 27.13 21.33 35.27 27.04] [53.32 29.51 21.73 34.77 25.11] [50.62 30.86 21.43 31.16 21.6] [55.98 34.63 29.37 25.38 21.95] [40.4 19.76 15.87 33.68 26.82] [49.61 33.91 22.77 30.3 20.6] [55.74 34.81 28.71 26.53 22.35] [50.14 14.68 21.48 24.1 34.19] [58.82 31.91 31.83 26.76 26.69] [52.51 34.99 24.94 28.94 21.07] [39.71 13.26 20.7 23.12 33.1]]'
SINR2_ture = [[9.85, 7.33, 4.98, 7.19, 8.61],
 [8.14, 5.79, 4.19, 6.53, 7.85],
 [7.43, 5.23, 3.69, 6.03, 7.26],
 [9.38, 6.89, 5.13, 7.51, 8.93],
 [7.81, 5.51, 4.01, 6.35, 7.67],
 [6.21, 4.73, 3.43, 5.87, 6.99],
 [6.58, 4.34, 3.21, 5.64, 6.76],
 [8.01, 5.71, 4.31, 6.83, 8.15],
 [6.45, 4.21, 3.51, 5.73, 6.85],
 [5.94, 4.01, 3.29, 5.59, 6.71],
 [7.93, 5.59, 4.25, 6.69, 8.11],
 [9.29, 6.92, 5.39, 7.13, 8.65],
 [8.57, 6.35, 4.71, 6.99, 8.21],
 [7.02, 5.07, 3.59, 6.21, 7.43],
 [5.46, 4.03, 3.23, 5.65, 6.77]]
SINR3 = '[[52.44 25.87 21.94 34.82 28.9] [53.79 30.14 20.79 34.2 23.26] [41.72 33.35 23.01 21.51 14.15] [36.75 17.11 19.6 28.03 33.76] [55.14 25.52 20.87 35.17 28.07] [53.94 29.37 21.8 34.83 25.34] [48.8 29.23 19.54 31.21 20.59] [52.04 34.86 28.92 24.8 21.02] [45.83 20.38 14.04 33.73 24.49] [54.57 33.68 23.11 31.13 21.56] [53.34 35.1 28.26 24.89 20.52] [49.57 15.48 22.0 24.59 34.59] [56.44 31.39 32.57 25.91 26.78] [53.4 34.83 25.57 29.32 21.96] [43.01 13.87 21.18 23.0 33.18]]'
SINR3_ture = [[9.62, 7.11, 4.83, 7.38, 8.79],
 [7.95, 5.63, 4.29, 6.71, 7.93],
 [7.32, 5.19, 3.83, 6.09, 7.31],
 [9.18, 6.81, 5.13, 7.49, 8.91],
 [7.71, 5.49, 4.09, 6.39, 7.71],
 [6.23, 4.73, 3.45, 5.91, 7.03],
 [6.61, 4.37, 3.25, 5.67, 6.79],
 [7.96, 5.71, 4.33, 6.85, 8.17],
 [6.49, 4.25, 3.55, 5.75, 6.87],
 [6.02, 4.07, 3.33, 5.65, 6.77],
 [7.95, 5.61, 4.27, 6.71, 8.13],
 [9.31, 6.95, 5.41, 7.15, 8.67],
 [8.59, 6.39, 4.73, 7.01, 8.23],
 [7.05, 5.11, 3.63, 6.25, 7.47],
 [5.51, 4.09, 3.27, 5.69, 6.81]]
SINR4 = '[[51.39 25.47 20.86 35.16 28.1] [48.92 30.46 20.48 33.51 22.3] [39.09 33.33 23.74 20.25 13.31] [47.47 16.43 19.72 27.44 33.8] [53.99 23.99 20.44 34.75 29.02] [51.02 29.73 21.66 34.67 24.8] [51.51 29.31 20.69 32.57 22.43] [52.67 35.04 28.07 23.75 19.44] [47.07 21.1 14.32 33.98 24.03] [54.59 33.36 23.45 31.76 22.46] [51.29 35.07 27.55 23.23 18.63] [49.66 16.02 22.3 25.02 34.8] [55.1 30.78 33.29 24.94 26.75] [54.64 34.71 25.88 29.51 22.43] [50.14 15.29 22.43 23.91 34.52]]'
SINR4_ture = [[9.43, 7.05, 4.73, 7.28, 8.69],
 [7.88, 5.55, 4.25, 6.69, 7.91],
 [7.31, 5.17, 3.83, 6.09, 7.31],
 [9.15, 6.83, 5.11, 7.47, 8.89],
 [7.73, 5.51, 4.11, 6.41, 7.73],
 [6.27, 4.75, 3.47, 5.93, 7.05],
 [6.66, 4.41, 3.29, 5.69, 6.81],
 [7.92, 5.73, 4.35, 6.87, 8.19],
 [6.52, 4.31, 3.61, 5.79, 6.91],
 [6.08, 4.13, 3.39, 5.69, 6.81],
 [7.93, 5.63, 4.29, 6.73, 8.15],
 [9.29, 6.93, 5.41, 7.15, 8.67],
 [8.57, 6.41, 4.75, 7.03, 8.25],
 [7.06, 5.13, 3.65, 6.27, 7.49],
 [5.54, 4.11, 3.31, 5.71, 6.83]]
SINR5 = '[[53.77 24.56 19.02 35.41 26.74] [50.44 30.61 20.5 32.42 21.53] [39.34 33.69 26.77 19.86 15.8] [47.06 15.44 20.02 26.4 33.92] [54.17 22.27 19.8 34.06 29.88] [51.02 29.61 21.71 34.73 24.99] [47.7 28.44 19.83 32.9 22.26] [52.56 34.87 27.16 22.2 17.53] [49.15 21.9 14.74 34.22 23.74] [56.56 32.8 23.71 32.47 23.5] [50.55 34.85 27.23 22.15 17.54] [49.33 16.95 22.9 25.67 35.09] [56.12 29.95 34.06 23.72 26.59] [54.87 34.46 26.44 29.71 23.18] [50.32 16.54 23.14 24.87 35.04]]'
SINR5_ture = [[9.39, 7.01, 4.71, 7.26, 8.67],
 [7.85, 5.53, 4.23, 6.67, 7.89],
 [7.29, 5.15, 3.81, 6.07, 7.29],
 [9.12, 6.81, 5.09, 7.45, 8.87],
 [7.72, 5.49, 4.09, 6.39, 7.71],
 [6.26, 4.74, 3.46, 5.92, 7.04],
 [6.65, 4.41, 3.29, 5.69, 6.81],
 [7.91, 5.73, 4.35, 6.87, 8.19],
 [6.57, 4.31, 3.61, 5.79, 6.91],
 [6.11, 4.13, 3.39, 5.69, 6.81],
 [7.92, 5.63, 4.29, 6.73, 8.15],
 [9.28, 6.92, 5.4, 7.14, 8.66],
 [8.56, 6.4, 4.74, 7.02, 8.24],
 [7.07, 5.12, 3.64, 6.26, 7.48],
 [5.55, 4.1, 3.3, 5.7, 6.82]]

def parse_string_data(data_str):
    # 移除多余的空格和换行符
    data_str = data_str.strip()
    # 使用正则表达式提取所有数字
    numbers = re.findall(r'[-+]?\d*\.?\d+', data_str)
    # 转换为浮点数
    numbers = [float(num) for num in numbers]
    # 重新reshape为15x5的数组
    return np.array(numbers).reshape(15, 5)

# 解析所有SINR数据
SINR1_pred = parse_string_data(SINR1)
SINR2_pred = parse_string_data(SINR2)
SINR3_pred = parse_string_data(SINR3)
SINR4_pred = parse_string_data(SINR4)
SINR5_pred = parse_string_data(SINR5)

# 将true数据转换为numpy数组
SINR1_true = np.array(SINR1_ture)
SINR2_true = np.array(SINR2_ture)
SINR3_true = np.array(SINR3_ture)
SINR4_true = np.array(SINR4_ture)
SINR5_true = np.array(SINR5_ture)

mae_sinr1 = mae(SINR1_true, SINR1_pred)
mae_sinr2 = mae(SINR2_true, SINR2_pred)
mae_sinr3 = mae(SINR3_true, SINR3_pred)
mae_sinr4 = mae(SINR4_true, SINR4_pred)
mae_sinr5 = mae(SINR5_true, SINR5_pred)

# 计算平均MAE
all_mae = [mae_sinr1, mae_sinr2, mae_sinr3, mae_sinr4, mae_sinr5]
average_mae = np.mean(all_mae)
print(average_mae)

24.755786666666662


In [11]:
from utils_new import mae, cosine_similarity, accuracy
import numpy as np
import re

# Room 2-1
SINR1 = '[[49.7 18.7 33.26 16.02 28.06] [50.11 31.29 30.45 18.71 18.28] [48.45 33.07 25.96 16.71 12.97] [57.97 22.71 22.38 31.01 30.48] [48.6 14.69 16.82 28.05 32.22] [48.73 15.8 13.46 32.04 27.75] [57.27 23.56 29.94 24.21 30.86] [50.67 17.51 30.81 17.71 31.21] [40.21 17.53 33.16 11.83 24.17] [53.2 30.16 31.72 18.79 19.63] [53.86 31.87 29.93 22.06 20.9] [43.07 10.86 20.35 18.62 34.07] [48.39 34.25 23.64 18.44 12.85] [50.16 30.19 30.22 15.04 15.05] [51.96 34.18 19.9 26.72 15.96] [50.58 18.88 33.48 15.92 27.73] [38.79 32.6 24.08 15.13 -16.26] [37.3 16.62 -16.96 32.39 18.15] [48.13 19.15 12.91 34.54 22.9] [52.6 34.96 24.26 23.26 16.79]]'
SINR1_ture = [[9.81, 7.34, 5.21, 7.59, 8.91],
 [8.14, 5.81, 4.31, 6.73, 7.95],
 [7.43, 5.25, 3.93, 6.11, 7.33],
 [9.38, 6.92, 5.14, 7.51, 8.93],
 [7.85, 5.62, 4.25, 6.69, 7.91],
 [6.28, 4.78, 3.49, 5.93, 7.05],
 [7.65, 5.38, 4.09, 6.43, 7.75],
 [9.02, 6.71, 5.03, 7.39, 8.81],
 [8.45, 6.22, 4.67, 7.01, 8.23],
 [7.09, 5.05, 3.83, 6.25, 7.47],
 [6.52, 4.62, 3.45, 5.89, 7.01],
 [8.92, 6.59, 4.93, 7.29, 8.71],
 [9.25, 7.03, 5.39, 7.65, 8.97],
 [8.59, 6.37, 4.75, 7.05, 8.27],
 [7.15, 5.13, 3.69, 6.31, 7.53],
 [5.59, 4.25, 3.31, 5.73, 6.85],
 [7.93, 5.67, 4.33, 6.87, 8.19],
 [9.46, 7.15, 5.49, 7.83, 9.15],
 [8.83, 6.59, 4.95, 7.31, 8.73],
 [7.31, 5.21, 3.93, 6.41, 7.73]]
SINR2 = '[[51.24 17.66 32.71 15.31 28.23] [52.55 31.25 30.29 18.16 17.67] [47.3 32.47 27.23 16.39 13.63] [56.97 22.63 22.85 30.52 30.87] [48.32 14.91 16.75 28.41 32.0] [48.8 15.68 13.89 31.68 28.37] [58.88 24.82 29.93 24.84 29.96] [52.9 18.85 31.66 18.35 30.69] [50.37 17.56 33.39 12.85 25.22] [51.78 30.59 31.37 19.24 19.66] [54.43 32.67 28.68 23.27 20.78] [49.11 11.56 21.93 18.35 34.16] [47.9 34.42 23.21 18.84 12.87] [48.64 30.72 29.64 15.32 14.75] [51.51 33.75 19.44 27.5 16.18] [49.88 19.34 34.08 15.42 26.56] [37.86 32.6 25.24 15.08 -16.43] [31.93 16.77 -17.44 31.95 17.89] [51.3 19.95 14.3 34.66 24.15] [52.16 34.84 23.67 24.73 17.46]]'
SINR2_ture = [[9.73, 7.39, 5.29, 7.64, 8.96],
 [8.22, 5.91, 4.43, 6.83, 7.99],
 [7.58, 5.31, 3.99, 6.19, 7.41],
 [9.51, 6.99, 5.19, 7.57, 8.99],
 [8.04, 5.73, 4.37, 6.79, 8.01],
 [6.46, 4.89, 3.59, 5.99, 7.11],
 [7.81, 5.51, 4.23, 6.55, 7.87],
 [9.14, 6.83, 5.11, 7.47, 8.89],
 [8.55, 6.45, 4.85, 7.11, 8.33],
 [7.15, 5.15, 3.89, 6.31, 7.53],
 [6.62, 4.71, 3.53, 5.95, 7.07],
 [8.96, 6.65, 4.99, 7.33, 8.75],
 [9.31, 7.11, 5.47, 7.79, 9.11],
 [8.67, 6.49, 4.91, 7.19, 8.51],
 [7.27, 5.23, 3.83, 6.43, 7.75],
 [5.73, 4.31, 3.37, 5.79, 6.91],
 [8.01, 5.73, 4.39, 6.91, 8.23],
 [9.48, 7.21, 5.59, 7.91, 9.23],
 [8.87, 6.69, 5.03, 7.39, 8.81],
 [7.39, 5.31, 4.01, 6.51, 7.83]]
SINR3 = '[[50.86 17.51 32.62 15.2 28.24] [51.84 31.23 30.16 17.79 17.25] [48.27 32.23 27.66 16.28 13.87] [55.05 22.56 23.13 30.23 31.1] [51.61 14.86 16.78 28.32 32.06] [48.69 15.33 14.98 30.55 29.88] [62.85 26.46 29.81 25.46 28.61] [50.4 19.99 32.29 18.82 30.13] [50.42 17.82 33.37 14.23 26.67] [51.99 31.29 30.72 20.08 19.77] [55.39 33.35 27.13 24.64 20.54] [49.83 12.25 23.4 18.12 34.06] [49.39 34.7 22.43 19.62 12.93] [50.86 31.44 28.73 15.6 14.12] [53.1 33.03 18.89 28.65 16.63] [51.33 19.6 34.51 14.73 25.29] [35.52 32.64 26.07 15.1 -16.56] [10.52 16.95 -17.93 31.52 18.62] [50.96 20.8 15.79 34.56 25.38] [54.37 34.62 23.04 25.82 17.81]]'
SINR3_ture = [[9.69, 7.42, 5.33, 7.66, 8.98],
 [8.26, 5.96, 4.48, 6.88, 8.02],
 [7.63, 5.36, 4.02, 6.22, 7.44],
 [9.56, 7.04, 5.22, 7.6, 9.02],
 [8.11, 5.77, 4.43, 6.83, 8.05],
 [6.53, 4.94, 3.65, 6.01, 7.13],
 [7.88, 5.58, 4.28, 6.6, 7.92],
 [9.19, 6.9, 5.16, 7.52, 8.94],
 [8.62, 6.52, 4.94, 7.16, 8.38],
 [7.25, 5.23, 3.95, 6.37, 7.59],
 [6.74, 4.81, 3.57, 6.01, 7.13],
 [9.03, 6.75, 5.01, 7.35, 8.77],
 [9.38, 7.14, 5.51, 7.83, 9.15],
 [8.75, 6.59, 5.01, 7.31, 8.63],
 [7.37, 5.29, 4.01, 6.53, 7.85],
 [5.85, 4.39, 3.43, 5.83, 6.95],
 [8.11, 5.83, 4.43, 6.95, 8.27],
 [9.56, 7.27, 5.67, 7.99, 9.21],
 [8.97, 6.79, 5.11, 7.43, 8.85],
 [7.51, 5.39, 4.09, 6.61, 7.93]]
SINR4 = '[[48.02 16.52 31.68 14.6 28.43] [49.6 31.07 29.96 16.97 16.41] [49.78 32.13 27.83 16.22 13.96] [56.64 22.59 23.08 30.29 31.04] [48.41 15.21 16.58 28.93 31.62] [50.95 15.02 15.49 29.77 30.67] [61.55 27.44 29.65 25.73 27.73] [52.26 21.3 32.83 19.38 29.41] [50.39 17.91 33.26 14.7 27.18] [55.44 32.24 29.64 21.27 19.78] [55.8 33.51 26.52 25.17 20.44] [49.21 12.63 24.33 17.91 33.83] [50.32 34.91 21.27 20.82 12.99] [49.45 32.05 27.78 15.87 13.56] [50.22 32.72 18.72 29.1 16.86] [48.61 19.75 34.69 14.37 24.58] [42.3 32.59 24.52 15.1 -16.37] [38.87 17.1 -17.36 31.97 17.99] [50.13 21.18 16.76 34.29 26.28] [53.76 34.33 22.33 26.75 17.97]]'
SINR4_ture = [[9.63, 7.41, 5.32, 7.65, 8.97],
 [8.3, 5.94, 4.46, 6.86, 8.0],
 [7.67, 5.36, 4.01, 6.21, 7.43],
 [9.59, 7.03, 5.19, 7.59, 9.01],
 [8.14, 5.76, 4.42, 6.82, 8.04],
 [6.56, 4.94, 3.64, 6.0, 7.12],
 [7.91, 5.59, 4.27, 6.59, 7.91],
 [9.22, 6.9, 5.15, 7.51, 8.93],
 [8.65, 6.51, 4.93, 7.15, 8.37],
 [7.28, 5.22, 3.94, 6.36, 7.58],
 [6.79, 4.8, 3.56, 6.0, 7.12],
 [9.05, 6.74, 5.0, 7.34, 8.76],
 [9.4, 7.13, 5.5, 7.82, 9.14],
 [8.77, 6.58, 5.0, 7.3, 8.62],
 [7.39, 5.28, 4.0, 6.52, 7.84],
 [5.87, 4.38, 3.42, 5.82, 6.94],
 [8.13, 5.83, 4.43, 6.93, 8.25],
 [9.58, 7.27, 5.66, 7.98, 9.2],
 [8.99, 6.79, 5.11, 7.43, 8.85],
 [7.54, 5.39, 4.09, 6.61, 7.93]]
SINR5 = '[[47.23 16.47 30.82 14.86 28.44] [50.43 30.76 29.7 15.62 15.07] [47.57 32.43 27.33 16.4 13.72] [53.98 22.59 23.09 30.28 31.05] [52.06 15.46 16.45 29.35 31.3] [48.82 14.55 16.09 28.66 31.62] [62.51 28.83 29.28 25.99 26.38] [55.31 22.53 33.25 19.77 28.57] [51.4 17.95 33.24 14.82 27.3] [53.33 33.22 28.24 22.39 19.46] [55.46 33.79 25.13 26.21 20.05] [47.92 12.89 25.11 17.66 33.52] [48.82 34.93 20.76 21.37 13.01] [48.98 32.48 26.96 16.09 13.1] [52.55 32.32 18.43 29.6 17.03] [50.62 19.99 34.86 13.83 23.49] [40.6 32.69 23.38 15.3 -16.07] [30.59 17.13 -16.69 32.52 17.98] [51.89 21.44 17.85 33.79 27.37] [52.08 34.07 21.96 27.37 18.16]]'
SINR5_ture = [[9.59, 7.39, 5.31, 7.64, 8.96],
 [8.29, 5.93, 4.45, 6.85, 8.01],
 [7.66, 5.37, 4.01, 6.21, 7.43],
 [9.58, 7.04, 5.19, 7.59, 9.01],
 [8.15, 5.77, 4.43, 6.83, 8.05],
 [6.57, 4.95, 3.65, 6.01, 7.13],
 [7.92, 5.6, 4.29, 6.61, 7.93],
 [9.23, 6.91, 5.15, 7.51, 8.93],
 [8.66, 6.53, 4.95, 7.15, 8.37],
 [7.31, 5.23, 3.95, 6.37, 7.59],
 [6.82, 4.81, 3.57, 6.01, 7.13],
 [9.06, 6.75, 5.01, 7.35, 8.77],
 [9.41, 7.15, 5.51, 7.83, 9.15],
 [8.78, 6.61, 5.01, 7.31, 8.63],
 [7.41, 5.29, 4.01, 6.53, 7.85],
 [5.89, 4.41, 3.43, 5.83, 6.95],
 [8.15, 5.83, 4.43, 6.95, 8.27],
 [9.6, 7.29, 5.69, 7.99, 9.21],
 [9.01, 6.81, 5.11, 7.43, 8.85],
 [7.57, 5.41, 4.11, 6.63, 7.95]]

def parse_string_data(data_str):
    # 移除多余的空格和换行符
    data_str = data_str.strip()
    # 使用正则表达式提取所有数字
    numbers = re.findall(r'[-+]?\d*\.?\d+', data_str)
    # 转换为浮点数
    numbers = [float(num) for num in numbers]
    # 重新reshape为15x5的数组
    return np.array(numbers).reshape(20, 5)

# 解析所有SINR数据
SINR1_pred = parse_string_data(SINR1)
SINR2_pred = parse_string_data(SINR2)
SINR3_pred = parse_string_data(SINR3)
SINR4_pred = parse_string_data(SINR4)
SINR5_pred = parse_string_data(SINR5)

# 将true数据转换为numpy数组
SINR1_true = np.array(SINR1_ture)
SINR2_true = np.array(SINR2_ture)
SINR3_true = np.array(SINR3_ture)
SINR4_true = np.array(SINR4_ture)
SINR5_true = np.array(SINR5_ture)

mae_sinr1 = mae(SINR1_true, SINR1_pred)
mae_sinr2 = mae(SINR2_true, SINR2_pred)
mae_sinr3 = mae(SINR3_true, SINR3_pred)
mae_sinr4 = mae(SINR4_true, SINR4_pred)
mae_sinr5 = mae(SINR5_true, SINR5_pred)

# 计算平均MAE
all_mae = [mae_sinr1, mae_sinr2, mae_sinr3, mae_sinr4, mae_sinr5]
average_mae = np.mean(all_mae)
print(average_mae)

22.50942


In [12]:
from utils_new import mae, cosine_similarity, accuracy
import numpy as np
import re

# Room 3-1
SINR1 = '[[53.36 1.6 14.75 28.44 34.13 -1.66] [50.8 -18.52 26.07 27.5 17.55 18.36] [43.28 -6.87 33.71 22.94 -19.41 6.7] [38.38 -20.0 18.07 20.65 14.37 33.58] [58.91 8.01 26.35 33.28 23.7 -8.05] [48.53 -20.0 26.27 24.14 13.22 19.79] [24.39 17.86 -20.0 19.95 31.36 -20.0] [22.52 -1.4 -20.0 16.0 30.81 0.91] [54.56 -16.86 24.61 29.81 21.24 16.76] [50.23 -16.66 28.76 25.62 13.93 16.46] [53.8 14.53 19.04 29.79 27.65 -14.62] [52.82 8.28 31.51 29.66 16.86 -8.34] [31.19 16.79 30.84 17.6 -20.0 -20.0] [58.18 -9.02 28.47 32.29 21.07 8.98] [18.08 -20.0 -20.0 17.69 13.85 30.73] [50.79 0.56 34.34 27.96 14.25 -0.62] [25.59 11.16 -20.0 16.19 30.27 -19.32] [46.59 17.8 12.3 24.15 27.99 -18.1] [48.05 -20.0 26.02 25.16 14.8 19.75] [55.11 -12.32 22.86 31.94 25.58 12.26]]'
SINR1_ture = [[9.86, 7.41, 5.33, 7.67, 8.99, 7.93],
 [8.23, 5.97, 4.51, 6.93, 8.03, 7.01],
 [7.61, 5.41, 4.13, 6.35, 7.47, 6.49],
 [9.62, 7.09, 5.23, 7.61, 9.03, 7.93],
 [8.11, 5.83, 4.47, 6.91, 8.07, 7.03],
 [6.55, 5.01, 3.75, 6.11, 7.13, 6.37],
 [7.94, 5.71, 4.43, 6.87, 7.99, 7.01],
 [9.28, 7.01, 5.21, 7.59, 9.01, 8.01],
 [8.63, 6.59, 4.99, 7.23, 8.35, 7.39],
 [7.29, 5.31, 4.03, 6.37, 7.59, 6.63],
 [6.83, 5.01, 3.79, 6.11, 7.13, 6.37],
 [9.08, 6.81, 5.11, 7.45, 8.87, 7.99],
 [9.43, 7.21, 5.61, 7.91, 9.23, 8.33],
 [8.81, 6.67, 5.13, 7.41, 8.73, 7.93],
 [7.43, 5.37, 4.11, 6.55, 7.87, 7.01],
 [6.01, 4.79, 3.61, 5.99, 7.01, 6.25],
 [8.23, 5.93, 4.55, 6.97, 8.19, 7.29],
 [9.58, 7.39, 5.81, 8.01, 9.31, 8.51],
 [8.97, 6.83, 5.23, 7.61, 8.93, 8.13],
 [7.59, 5.51, 4.25, 6.69, 7.99, 7.23]]
SINR2 = '[[52.13 1.75 13.81 27.52 34.46 -1.82] [51.31 -17.58 26.59 28.05 17.83 17.44] [48.45 -7.74 33.53 23.44 -18.79 7.57] [36.14 -20.0 18.99 20.69 13.14 33.07] [61.29 5.96 26.78 33.68 23.78 -5.99] [50.45 -19.61 26.64 24.49 13.54 19.31] [30.02 4.05 -20.0 19.25 32.49 -4.38] [32.05 -3.31 -20.0 16.0 30.6 2.79] [54.85 -17.4 23.15 29.71 22.42 17.29] [48.89 -16.87 28.83 24.67 12.69 16.62] [52.36 16.42 18.45 28.8 26.9 -16.54] [54.65 6.96 31.09 30.93 18.24 -7.01] [32.33 19.12 31.26 18.97 -20.0 -20.0] [58.57 -8.7 27.73 32.73 22.04 8.66] [29.28 -20.0 -20.0 15.41 12.71 30.07] [52.12 0.28 34.3 28.09 14.38 -0.33] [30.07 10.56 -20.0 13.85 27.95 -20.0] [48.0 15.54 12.1 24.47 29.62 -15.78] [49.08 -20.0 24.97 24.24 14.22 21.01] [56.54 -11.17 24.47 32.5 24.55 11.12]]'
SINR2_ture = [[9.89, 7.45, 5.37, 7.71, 8.99, 7.99],
 [8.31, 6.01, 4.55, 7.01, 8.03, 7.05],
 [7.69, 5.45, 4.17, 6.39, 7.51, 6.53],
 [9.73, 7.13, 5.27, 7.63, 9.05, 7.95],
 [8.24, 5.91, 4.53, 6.95, 8.09, 7.11],
 [6.69, 5.11, 3.81, 6.19, 7.21, 6.43],
 [8.04, 5.77, 4.47, 6.97, 8.19, 7.29],
 [9.41, 7.21, 5.63, 7.95, 9.27, 8.37],
 [8.86, 6.91, 5.23, 7.51, 8.93, 7.99],
 [7.51, 5.41, 4.13, 6.45, 7.67, 6.69],
 [6.99, 5.11, 3.85, 6.21, 7.23, 6.45],
 [9.15, 6.87, 5.17, 7.49, 8.91, 7.99],
 [9.5, 7.33, 5.73, 8.03, 9.35, 8.55],
 [8.97, 6.81, 5.23, 7.61, 8.93, 8.13],
 [7.59, 5.51, 4.25, 6.69, 7.99, 7.23],
 [6.11, 4.89, 3.71, 6.11, 7.13, 6.37],
 [8.35, 6.01, 4.59, 7.01, 8.19, 7.31],
 [9.68, 7.49, 5.89, 8.11, 9.41, 8.61],
 [9.09, 6.89, 5.29, 7.61, 8.93, 8.13],
 [7.71, 5.59, 4.31, 6.73, 8.01, 7.25]]
SINR3 = '[[51.59 1.88 12.94 26.63 34.68 -1.95] [49.53 -17.12 26.82 28.3 17.96 16.99] [47.68 -8.21 33.4 23.64 -18.51 8.04] [34.34 -20.0 19.51 20.73 12.35 32.59] [60.97 1.88 27.37 34.11 23.75 -1.91] [50.91 -18.18 27.52 25.66 14.58 17.97] [31.79 1.38 -20.0 19.23 33.11 -1.65] [31.43 -20.0 -20.0 17.36 30.8 16.36] [53.58 -17.79 21.06 29.25 24.01 17.67] [48.09 -16.85 28.91 24.25 12.04 16.58] [52.55 18.01 17.89 27.84 26.14 -18.16] [56.85 5.86 30.67 31.77 19.24 -5.9] [35.47 19.31 31.21 18.74 -20.0 -20.0] [57.65 -8.23 26.43 33.24 23.59 8.19] [27.2 -20.0 11.95 16.19 13.17 30.98] [52.38 0.73 34.35 27.9 14.2 -0.79] [35.26 -0.46 -20.0 15.73 30.16 -0.14] [48.36 12.89 11.73 24.58 31.25 -13.08] [42.23 -20.0 23.53 23.1 13.42 32.51] [60.17 -9.24 26.75 32.87 22.93 9.2]]'
SINR3_ture = [[9.92, 7.49, 5.41, 7.77, 9.01, 8.01],
 [8.37, 6.05, 4.59, 7.05, 8.07, 7.09],
 [7.75, 5.49, 4.23, 6.49, 7.61, 6.65],
 [9.79, 7.19, 5.33, 7.71, 9.13, 8.13],
 [8.34, 6.01, 4.55, 7.05, 8.09, 7.11],
 [6.81, 5.19, 3.89, 6.31, 7.43, 6.57],
 [8.13, 5.91, 4.53, 7.01, 8.23, 7.35],
 [9.51, 7.35, 5.79, 8.01, 9.33, 8.53],
 [9.04, 7.01, 5.29, 7.61, 8.93, 8.13],
 [7.67, 5.59, 4.29, 6.69, 7.99, 7.23],
 [7.11, 5.19, 3.89, 6.31, 7.43, 6.57],
 [9.28, 6.99, 5.29, 7.69, 8.91, 8.01],
 [9.63, 7.49, 5.89, 8.11, 9.43, 8.63],
 [9.16, 7.01, 5.31, 7.71, 8.93, 8.13],
 [7.79, 5.69, 4.39, 6.91, 8.03, 7.25],
 [6.33, 5.01, 3.81, 6.19, 7.31, 6.45],
 [8.57, 6.19, 4.71, 7.19, 8.41, 7.51],
 [9.88, 7.69, 6.01, 8.21, 9.51, 8.71],
 [9.29, 7.19, 5.59, 7.91, 9.13, 8.33],
 [7.93, 5.89, 4.51, 6.99, 8.11, 7.33]]
SINR4 = '[[49.27 2.02 11.73 25.34 34.83 -2.11] [51.43 -16.28 27.22 28.74 18.2 16.17] [50.33 -9.65 32.89 24.14 10.88 9.48] [33.97 -20.0 19.97 20.93 11.66 32.15] [61.7 -1.55 27.44 34.12 23.71 1.52] [48.31 -16.84 28.23 26.64 15.36 16.67] [30.29 -0.49 -20.0 19.27 33.21 0.24] [34.53 -20.0 -20.0 18.34 31.13 18.02] [51.87 -17.85 20.27 28.97 24.57 17.72] [49.82 -16.82 28.94 24.0 11.64 16.54] [51.24 19.5 17.34 26.86 25.27 -19.69] [57.22 4.28 30.17 32.57 20.28 -4.31] [20.86 7.97 31.24 18.43 -20.0 -8.38] [60.3 -7.67 25.61 33.48 24.6 7.64] [30.12 -20.0 14.3 17.87 13.53 32.22] [53.0 2.11 34.39 27.38 13.72 -2.17] [29.84 -2.21 -20.0 15.98 30.78 1.71] [50.42 9.9 11.12 24.32 32.72 -10.06] [29.82 -20.0 22.75 22.47 12.81 32.41] [58.69 -7.17 28.93 32.67 21.15 7.14]]'
SINR4_ture = [[9.95, 7.53, 5.45, 7.81, 9.03, 8.03],
 [8.42, 6.11, 4.63, 7.13, 8.15, 7.17],
 [7.81, 5.53, 4.27, 6.53, 7.65, 6.69],
 [9.81, 7.23, 5.37, 7.75, 9.17, 8.17],
 [8.42, 6.07, 4.61, 7.13, 8.15, 7.17],
 [7.01, 5.23, 3.97, 6.33, 7.45, 6.59],
 [8.15, 5.93, 4.55, 7.07, 8.29, 7.41],
 [9.55, 7.41, 5.85, 8.07, 9.39, 8.59],
 [9.08, 7.01, 5.31, 7.71, 8.93, 8.13],
 [7.73, 5.61, 4.33, 6.75, 7.97, 7.21],
 [7.19, 5.23, 3.97, 6.33, 7.45, 6.59],
 [9.32, 6.99, 5.33, 7.73, 8.95, 8.05],
 [9.67, 7.53, 5.95, 8.15, 9.47, 8.67],
 [9.18, 7.03, 5.35, 7.75, 8.97, 8.17],
 [7.83, 5.71, 4.41, 6.91, 8.03, 7.25],
 [6.47, 5.07, 3.87, 6.27, 7.39, 6.53],
 [8.71, 6.29, 4.81, 7.29, 8.51, 7.61],
 [9.98, 7.79, 6.11, 8.31, 9.61, 8.81],
 [9.39, 7.29, 5.69, 7.99, 9.21, 8.41],
 [8.03, 5.91, 4.53, 6.99, 8.11, 7.33]]
SINR5 = '[[48.15 2.15 -18.62 24.05 34.78 -2.26] [52.41 -15.23 27.72 29.23 18.4 15.13] [49.99 -10.71 32.43 24.44 11.31 10.55] [31.38 -20.0 20.87 21.78 -13.53 31.6] [59.83 -4.11 27.39 33.89 23.5 4.08] [53.63 -15.59 28.77 27.5 16.01 15.46] [35.88 -2.74 -20.0 19.45 33.16 2.49] [30.81 -20.0 -20.0 17.11 30.85 16.23] [52.07 -17.92 19.42 28.58 25.11 17.79] [48.37 -16.82 28.94 23.92 11.5 16.52] [50.97 21.48 15.97 25.06 24.04 -20.0] [57.19 3.14 29.46 33.21 21.32 -3.17] [24.63 8.04 31.59 18.82 -20.0 -8.39] [61.58 -6.94 24.4 33.6 25.98 6.9] [38.36 -20.0 17.32 19.92 13.76 33.2] [48.89 3.06 34.34 27.01 13.4 -3.13] [36.43 -4.0 -20.0 16.95 31.31 3.59] [50.49 7.78 -18.29 23.9 33.52 -7.93] [26.71 -20.0 21.87 21.85 12.19 32.22] [57.97 -5.83 30.35 32.16 19.81 5.8]]'
SINR5_ture = [[9.98, 7.57, 5.49, 7.85, 9.07, 8.07],
 [8.47, 6.17, 4.67, 7.17, 8.19, 7.21],
 [7.87, 5.57, 4.31, 6.59, 7.71, 6.75],
 [9.85, 7.25, 5.41, 7.79, 9.21, 8.21],
 [8.47, 6.13, 4.67, 7.17, 8.19, 7.21],
 [7.09, 5.27, 3.99, 6.39, 7.51, 6.65],
 [8.23, 5.99, 4.59, 7.11, 8.33, 7.45],
 [9.63, 7.49, 5.91, 8.13, 9.45, 8.65],
 [9.16, 7.03, 5.35, 7.75, 8.97, 8.17],
 [7.83, 5.69, 4.41, 6.91, 8.03, 7.25],
 [7.31, 5.27, 3.99, 6.39, 7.51, 6.65],
 [9.42, 7.01, 5.33, 7.75, 8.97, 8.07],
 [9.77, 7.59, 6.01, 8.21, 9.53, 8.73],
 [9.28, 7.11, 5.47, 7.89, 9.21, 8.41],
 [7.97, 5.79, 4.51, 6.99, 8.11, 7.33],
 [6.63, 5.17, 3.89, 6.31, 7.43, 6.57],
 [8.95, 6.43, 4.93, 7.39, 8.61, 7.71],
 [10.0, 8.01, 6.31, 8.51, 10.0, 8.81],
 [9.49, 7.39, 5.81, 8.01, 9.33, 8.53],
 [8.17, 6.01, 4.63, 7.13, 8.35, 7.45]]

def parse_string_data(data_str):
    # 移除多余的空格和换行符
    data_str = data_str.strip()
    # 使用正则表达式提取所有数字
    numbers = re.findall(r'[-+]?\d*\.?\d+', data_str)
    # 转换为浮点数
    numbers = [float(num) for num in numbers]
    # 重新reshape为15x5的数组
    return np.array(numbers).reshape(20, 6)

# 解析所有SINR数据
SINR1_pred = parse_string_data(SINR1)
SINR2_pred = parse_string_data(SINR2)
SINR3_pred = parse_string_data(SINR3)
SINR4_pred = parse_string_data(SINR4)
SINR5_pred = parse_string_data(SINR5)

# 将true数据转换为numpy数组
SINR1_true = np.array(SINR1_ture)
SINR2_true = np.array(SINR2_ture)
SINR3_true = np.array(SINR3_ture)
SINR4_true = np.array(SINR4_ture)
SINR5_true = np.array(SINR5_ture)

mae_sinr1 = mae(SINR1_true, SINR1_pred)
mae_sinr2 = mae(SINR2_true, SINR2_pred)
mae_sinr3 = mae(SINR3_true, SINR3_pred)
mae_sinr4 = mae(SINR4_true, SINR4_pred)
mae_sinr5 = mae(SINR5_true, SINR5_pred)

# 计算平均MAE
all_mae = [mae_sinr1, mae_sinr2, mae_sinr3, mae_sinr4, mae_sinr5]
average_mae = np.mean(all_mae)
print(average_mae)

19.707733333333334


In [13]:
# 消融实验1: Proposed LLM vs LLM w/o Adapter
# 消融实验2: Proposed LLM vs LLM w/o Meta-Adapter
# 消融实验3: Proposed LLM vs MLP
from utils_new import mae, cosine_similarity, accuracy
import numpy as np
import re

# Room 1-1: 选择的5个测试样本的APS要有差异性，因此选了前5个trace的第一个sample，对于RA选择的5个sample也一样规则，SINR为了方便选的就是第一个trace的前5个samples
APS1 = [2, 1, 2, 4, 1, 1, 3, 1, 2, 4, 2, 4, 1, 2, 4]
APS1_ture = '[1 1 1 1 1 1 2 2 4 4 3 1 1 1 5]'
APS2 = [2, 4, 3, 1, 4, 1, 4, 1, 1, 2, 4, 1, 1, 2, 1, 4, 1, 4, 2, 4]
APS2_ture = '[2 5 4 1 3 1 3 1 2 1 5 1 4 1 2 4 3 3 3 1]'
APS3 = [2, 1, 4, 3, 1]
APS3_ture = '[1 1 5 1 4]'
APS4 = [1, 2, 3, 4, 1, 4, 2, 4, 1, 3]
APS4_ture = '[1 1 4 1 2 1 1 3 1 1]'
APS5 = [2, 1, 4, 4, 1, 4, 2, 4, 4, 3]
APS5_ture = '[3 1 1 1 5 1 1 1 2 4]'

def accuracy(pred_list, target_list):
    accuracy_list = [1 if p == t else 0 for p, t in (zip(pred_list, target_list))]
    accuracy_value = sum(accuracy_list) / len(accuracy_list) if len(accuracy_list) > 0 else 0
    return accuracy_list, accuracy_value

# 解析字符串格式的真实值
def parse_target_string(target_str):
    # 移除空格和方括号
    target_str = target_str.strip('[]')
    # 分割数字
    return [int(x) for x in target_str.split()]

# 解析所有真实值
APS1_true = parse_target_string(APS1_ture)
APS2_true = parse_target_string(APS2_ture)
APS3_true = parse_target_string(APS3_ture)
APS4_true = parse_target_string(APS4_ture)
APS5_true = parse_target_string(APS5_ture)

# 计算每个AP的准确率
acc1_list, acc1_value = accuracy(APS1, APS1_true)
acc2_list, acc2_value = accuracy(APS2, APS2_true)
acc3_list, acc3_value = accuracy(APS3, APS3_true)
acc4_list, acc4_value = accuracy(APS4, APS4_true)
acc5_list, acc5_value = accuracy(APS5, APS5_true)

# 计算平均准确率
all_acc_values = [acc1_value, acc2_value, acc3_value, acc4_value, acc5_value]
average_acc = np.mean(all_acc_values)

print(average_acc)

0.22666666666666666


In [14]:
# 消融实验1: Proposed LLM vs LLM w/o Adapter
# 消融实验2: Proposed LLM vs LLM w/o Meta-Adapter
# 消融实验3: Proposed LLM vs MLP
from utils_new import mae, cosine_similarity, accuracy
import numpy as np
import re

# Room 2-1: 选择的5个测试样本的APS要有差异性，因此选了前5个trace的第一个sample，对于RA选择的5个sample也一样规则，SINR为了方便选的就是第一个trace的前5个samples
APS1 = [1, 4, 3, 4, 4, 1, 4, 1, 4, 3, 1, 4, 3, 4, 2, 4, 2, 1, 2, 3]
APS1_ture = '[3 2 3 4 1 3 5 5 1 2 1 1 3 3 1 1 1 3 4 1]'
APS2 = [1, 2, 4, 4, 3, 1, 4, 2, 1, 4]
APS2_ture = '[1 1 2 1 4 1 1 1 1 3]'
APS3 = [1, 2, 4, 4, 1, 4, 3, 2, 1, 2, 2, 4, 2, 1, 4]
APS3_ture = '[5 2 1 4 1 1 5 1 3 1 1 2 1 1 3]'
APS4 = [2, 4, 3, 1, 4, 1, 4, 1, 4, 2]
APS4_ture = '[1 1 1 1 1 1 1 4 1 1]'
APS5 = [1, 4, 2, 4, 4, 4, 2, 1, 1, 2, 1, 2, 1, 2, 4]
APS5_ture = '[1 4 3 1 1 1 1 2 1 1 1 5 1 5 1]'

def accuracy(pred_list, target_list):
    accuracy_list = [1 if p == t else 0 for p, t in (zip(pred_list, target_list))]
    accuracy_value = sum(accuracy_list) / len(accuracy_list) if len(accuracy_list) > 0 else 0
    return accuracy_list, accuracy_value

# 解析字符串格式的真实值
def parse_target_string(target_str):
    # 移除空格和方括号
    target_str = target_str.strip('[]')
    # 分割数字
    return [int(x) for x in target_str.split()]

# 解析所有真实值
APS1_true = parse_target_string(APS1_ture)
APS2_true = parse_target_string(APS2_ture)
APS3_true = parse_target_string(APS3_ture)
APS4_true = parse_target_string(APS4_ture)
APS5_true = parse_target_string(APS5_ture)

# 计算每个AP的准确率
acc1_list, acc1_value = accuracy(APS1, APS1_true)
acc2_list, acc2_value = accuracy(APS2, APS2_true)
acc3_list, acc3_value = accuracy(APS3, APS3_true)
acc4_list, acc4_value = accuracy(APS4, APS4_true)
acc5_list, acc5_value = accuracy(APS5, APS5_true)

# 计算平均准确率
all_acc_values = [acc1_value, acc2_value, acc3_value, acc4_value, acc5_value]
average_acc = np.mean(all_acc_values)

print(average_acc)

0.25999999999999995


In [15]:
# 消融实验1: Proposed LLM vs LLM w/o Adapter
# 消融实验2: Proposed LLM vs LLM w/o Meta-Adapter
# 消融实验3: Proposed LLM vs MLP
from utils_new import mae, cosine_similarity, accuracy
import numpy as np
import re

# Room 3-1: 选择的5个测试样本的APS要有差异性，因此选了前5个trace的第一个sample，对于RA选择的5个sample也一样规则，SINR为了方便选的就是第一个trace的前5个samples
APS1 = [1, 5, 3, 2, 4, 5, 1, 1, 6, 3, 1, 3, 1, 6, 4, 2, 1, 2, 6, 4]
APS1_ture = '[5 1 3 6 4 1 5 5 6 3 4 1 2 1 6 1 5 1 1 1]'
APS2 = [1, 5, 2, 4, 3, 5, 1, 1, 6, 2, 1, 3, 1, 6, 4, 2, 1, 2, 6, 4]
APS2_ture = '[3 1 4 1 1 5 1 6 2 1 1 2 2 1 3 4 6 1 5 1]'
APS3 = [1, 5, 2, 4, 3, 5, 1, 1, 6, 2, 1, 3, 1, 6, 4]
APS3_ture = '[6 3 4 1 1 3 5 1 1 2 6 1 2 1 5]'
APS4 = [1, 4, 2, 5, 3, 5, 1, 4, 6, 2]
APS4_ture = '[1 5 4 1 6 1 1 1 3 1]'
APS5 = [2, 1, 5, 4, 3, 5, 1, 4, 2, 1]
APS5_ture = '[1 1 2 4 1 3 1 1 1 1]'

def accuracy(pred_list, target_list):
    accuracy_list = [1 if p == t else 0 for p, t in (zip(pred_list, target_list))]
    accuracy_value = sum(accuracy_list) / len(accuracy_list) if len(accuracy_list) > 0 else 0
    return accuracy_list, accuracy_value

# 解析字符串格式的真实值
def parse_target_string(target_str):
    # 移除空格和方括号
    target_str = target_str.strip('[]')
    # 分割数字
    return [int(x) for x in target_str.split()]

# 解析所有真实值
APS1_true = parse_target_string(APS1_ture)
APS2_true = parse_target_string(APS2_ture)
APS3_true = parse_target_string(APS3_ture)
APS4_true = parse_target_string(APS4_ture)
APS5_true = parse_target_string(APS5_ture)

# 计算每个AP的准确率
acc1_list, acc1_value = accuracy(APS1, APS1_true)
acc2_list, acc2_value = accuracy(APS2, APS2_true)
acc3_list, acc3_value = accuracy(APS3, APS3_true)
acc4_list, acc4_value = accuracy(APS4, APS4_true)
acc5_list, acc5_value = accuracy(APS5, APS5_true)

# 计算平均准确率
all_acc_values = [acc1_value, acc2_value, acc3_value, acc4_value, acc5_value]
average_acc = np.mean(all_acc_values)

print(average_acc)

0.2166666666666667


In [21]:
# 消融实验1: Proposed LLM vs LLM w/o Adapter
# 消融实验2: Proposed LLM vs LLM w/o Meta-Adapter
# 消融实验3: Proposed LLM vs MLP
from utils_new import mae, cosine_similarity, accuracy
import numpy as np

# Room 1-1: 选择的5个测试样本的APS要有差异性，因此选了前5个trace的第一个sample，对于RA选择的5个sample也一样规则，SINR为了方便选的就是第一个trace的前5个samples
RA1 = [[0.9, 0.1, 0.0, 0.0, 0.0], [0.0, 0.8, 0.2, 0.0, 0.0], [0.0, 0.0, 0.0, 0.9, 0.1], [0.0, 0.0, 0.0, 0.0, 1.0], [0.7, 0.3, 0.0, 0.0, 0.0], [0.0, 0.8, 0.2, 0.0, 0.0], [0.0, 0.0, 0.0, 0.9, 0.1], [0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 0.9, 0.1, 0.0], [0.0, 0.8, 0.2, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0], [0.9, 0.1, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 0.0, 0.9, 0.1], [0.0, 0.0, 0.0, 0.0, 1.0]]
RA1_ture = '[[0.03 0.0 0.0 0.0 0.0] [0.03 0.0 0.0 0.0 0.0] [0.22 0.0 0.0 0.0 0.0] [0.12 0.0 0.0 0.0 0.0] [0.13 0.0 0.0 0.0 0.0] [0.16 0.0 0.0 0.0 0.0] [0.0 0.56 0.0 0.0 0.0] [0.0 0.44 0.0 0.0 0.0] [0.0 0.0 0.0 0.39 0.0] [0.0 0.0 0.0 0.61 0.0] [0.0 0.0 1.0 0.0 0.0] [0.06 0.0 0.0 0.0 0.0] [0.15 0.0 0.0 0.0 0.0] [0.11 0.0 0.0 0.0 0.0] [0.0 0.0 0.0 0.0 1.0]]'
RA2 = [[0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 1.0, 0.0, 0.0]]
RA2_ture = '[[0.0 0.33 0.0 0.0 0.0] [0.0 0.0 0.0 0.0 0.5] [0.0 0.0 0.0 0.33 0.0] [0.18 0.0 0.0 0.0 0.0] [0.0 0.0 0.2 0.0 0.0] [0.11 0.0 0.0 0.0 0.0] [0.0 0.0 0.2 0.0 0.0] [0.06 0.0 0.0 0.0 0.0] [0.0 0.33 0.0 0.0 0.0] [0.14 0.0 0.0 0.0 0.0] [0.0 0.0 0.0 0.0 0.5] [0.17 0.0 0.0 0.0 0.0] [0.0 0.0 0.0 0.33 0.0] [0.14 0.0 0.0 0.0 0.0] [0.0 0.34 0.0 0.0 0.0] [0.0 0.0 0.0 0.33 0.0] [0.0 0.0 0.2 0.0 0.0] [0.0 0.0 0.2 0.0 0.0] [0.0 0.0 0.2 0.0 0.0] [0.19 0.0 0.0 0.0 0.0]]'
RA3 = [[1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0]]
RA3_ture = '[[0.15 0.0 0.0 0.0 0.0] [0.21 0.0 0.0 0.0 0.0] [0.0 0.0 0.0 0.0 0.79] [0.51 0.0 0.0 0.0 0.0] [0.0 0.0 0.0 0.72 0.0]]'
RA4 = [[1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0], [1.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0]]
RA4_ture = '[[[0.14 0.0 0.0 0.0 0.0] [0.35 0.0 0.0 0.0 0.0] [0.0 0.0 0.0 0.82 0.0] [0.05 0.0 0.0 0.0 0.0] [0.0 1.0 0.0 0.0 0.0] [0.14 0.0 0.0 0.0 0.0] [0.1 0.0 0.0 0.0 0.0] [0.0 0.0 0.83 0.0 0.0] [0.03 0.0 0.0 0.0 0.0] [0.2 0.0 0.0 0.0 0.0]]'
RA5= [[0.0, 1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0], [1.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0]]
RA5_ture = '[[0.0 0.0 0.96 0.0 0.0] [0.17 0.0 0.0 0.0 0.0] [0.28 0.0 0.0 0.0 0.0] [0.08 0.0 0.0 0.0 0.0] [0.0 0.0 0.0 0.0 0.99] [0.11 0.0 0.0 0.0 0.0] [0.16 0.0 0.0 0.0 0.0] [0.2 0.0 0.0 0.0 0.0] [0.0 1.0 0.0 0.0 0.0] [0.0 0.0 0.0 1.0 0.0]]'

def cosine_similarity(list1, list2):
    vec1 = np.array(list1)
    vec2 = np.array(list2)
    if vec1.ndim == 1 and vec2.ndim == 1:
        dot_product = np.dot(vec1, vec2)
        norm1 = np.linalg.norm(vec1)
        norm2 = np.linalg.norm(vec2)
        if norm1 == 0 or norm2 == 0:
            return 0
        return dot_product / (norm1 * norm2)
    elif vec1.ndim == 2 and vec2.ndim == 2 and vec1.shape == vec2.shape:
        dot_products = np.sum(vec1 * vec2, axis=1)
        norms1 = np.linalg.norm(vec1, axis=1)
        norms2 = np.linalg.norm(vec2, axis=1)
        norms_product = norms1 * norms2
        norms_product[norms_product == 0] = 1
        similarities = dot_products / norms_product
        return np.mean(similarities)
    else:
        raise ValueError("输入必须是相同形状的一维向量或二维数组。")

# 解析字符串格式的真实值
def parse_ra_target(target_str):
    # 移除空格和方括号
    target_str = target_str.strip()
    # 使用正则表达式提取所有数字
    numbers = re.findall(r'[-+]?\d*\.?\d+', target_str)
    # 转换为浮点数
    numbers = [float(num) for num in numbers]
    # 根据数据推断形状
    if target_str.count('[') == 2:  # 2D数组
        # 计算每行有多少列（假设是5列）
        num_elements = len(numbers)
        num_rows = num_elements // 5
        return np.array(numbers).reshape(num_rows, 5)
    else:
        return np.array(numbers).reshape(-1, 5)

# 解析所有真实值
RA1_true = parse_ra_target(RA1_ture)
RA2_true = parse_ra_target(RA2_ture)
RA3_true = parse_ra_target(RA3_ture)
RA4_true = parse_ra_target(RA4_ture)
RA5_true = parse_ra_target(RA5_ture)

# 计算每个RA的余弦相似度
cos_sim1 = cosine_similarity(RA1, RA1_true)
cos_sim2 = cosine_similarity(RA2, RA2_true)
cos_sim3 = cosine_similarity(RA3, RA3_true)
cos_sim4 = cosine_similarity(RA4, RA4_true)
cos_sim5 = cosine_similarity(RA5, RA5_true)

# 计算平均余弦相似度
all_cos_sim = [cos_sim1, cos_sim2, cos_sim3, cos_sim4, cos_sim5]
average_cos_sim = np.mean(all_cos_sim)

print(average_cos_sim)

0.4235645870058685


In [22]:
# 消融实验1: Proposed LLM vs LLM w/o Adapter
# 消融实验2: Proposed LLM vs LLM w/o Meta-Adapter
# 消融实验3: Proposed LLM vs MLP
from utils_new import mae, cosine_similarity, accuracy
import numpy as np

# Room 2-1: 选择的5个测试样本的APS要有差异性，因此选了前5个trace的第一个sample，对于RA选择的5个sample也一样规则，SINR为了方便选的就是第一个trace的前5个samples
RA1 = [[0.0, 0.0, 0.0, 1.0, 0.0], [0.0, 1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 1.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 1.0, 0.0, 0.0]]
RA1_ture = '[[0.0 0.0 0.17 0.0 0.0] [0.0 0.45 0.0 0.0 0.0] [0.0 0.0 0.17 0.0 0.0] [0.0 0.0 0.0 0.46 0.0] [0.03 0.0 0.0 0.0 0.0] [0.0 0.0 0.17 0.0 0.0] [0.0 0.0 0.0 0.0 0.5] [0.0 0.0 0.0 0.0 0.5] [0.12 0.0 0.0 0.0 0.0] [0.0 0.55 0.0 0.0 0.0] [0.26 0.0 0.0 0.0 0.0] [0.06 0.0 0.0 0.0 0.0] [0.0 0.0 0.17 0.0 0.0] [0.0 0.0 0.17 0.0 0.0] [0.05 0.0 0.0 0.0 0.0] [0.06 0.0 0.0 0.0 0.0] [0.09 0.0 0.0 0.0 0.0] [0.0 0.0 0.17 0.0 0.0] [0.0 0.0 0.0 0.5 0.0] [0.31 0.0 0.0 0.0 0.0]]'
RA2 = [[0.0, 1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0], [1.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0]]
RA2_ture = '[[0.2 0.0 0.0 0.0 0.0] [0.04 0.0 0.0 0.0 0.0] [0.0 0.71 0.0 0.0 0.0] [0.24 0.0 0.0 0.0 0.0] [0.0 0.0 0.0 0.9 0.0] [0.24 0.0 0.0 0.0 0.0] [0.15 0.0 0.0 0.0 0.0] [0.16 0.0 0.0 0.0 0.0] [0.33 0.0 0.0 0.0 0.0] [0.0 0.0 0.93 0.0 0.0]]'
RA3 = [[0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0], [1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0], [1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 0.0, 0.0, 1.0]]
RA3_ture = '[[0.0 0.0 0.0 0.0 0.49] [0.0 0.75 0.0 0.0 0.0] [0.2 0.0 0.0 0.0 0.0] [0.0 0.0 0.0 0.8 0.0] [0.11 0.0 0.0 0.0 0.0] [0.16 0.0 0.0 0.0 0.0] [0.0 0.0 0.0 0.0 0.51] [0.27 0.0 0.0 0.0 0.0] [0.0 0.0 0.5 0.0 0.0] [0.03 0.0 0.0 0.0 0.0] [0.12 0.0 0.0 0.0 0.0] [0.0 0.23 0.0 0.0 0.0] [0.05 0.0 0.0 0.0 0.0] [0.03 0.0 0.0 0.0 0.0] [0.0 0.0 0.5 0.0 0.0]]'


def cosine_similarity(list1, list2):
    vec1 = np.array(list1)
    vec2 = np.array(list2)
    if vec1.ndim == 1 and vec2.ndim == 1:
        dot_product = np.dot(vec1, vec2)
        norm1 = np.linalg.norm(vec1)
        norm2 = np.linalg.norm(vec2)
        if norm1 == 0 or norm2 == 0:
            return 0
        return dot_product / (norm1 * norm2)
    elif vec1.ndim == 2 and vec2.ndim == 2 and vec1.shape == vec2.shape:
        dot_products = np.sum(vec1 * vec2, axis=1)
        norms1 = np.linalg.norm(vec1, axis=1)
        norms2 = np.linalg.norm(vec2, axis=1)
        norms_product = norms1 * norms2
        norms_product[norms_product == 0] = 1
        similarities = dot_products / norms_product
        return np.mean(similarities)
    else:
        raise ValueError("输入必须是相同形状的一维向量或二维数组。")

# 解析字符串格式的真实值
def parse_ra_target(target_str):
    # 移除空格和方括号
    target_str = target_str.strip()
    # 使用正则表达式提取所有数字
    numbers = re.findall(r'[-+]?\d*\.?\d+', target_str)
    # 转换为浮点数
    numbers = [float(num) for num in numbers]
    # 根据数据推断形状
    if target_str.count('[') == 2:  # 2D数组
        # 计算每行有多少列（假设是5列）
        num_elements = len(numbers)
        num_rows = num_elements // 5
        return np.array(numbers).reshape(num_rows, 5)
    else:
        return np.array(numbers).reshape(-1, 5)

# 解析所有真实值
RA1_true = parse_ra_target(RA1_ture)
RA2_true = parse_ra_target(RA2_ture)
RA3_true = parse_ra_target(RA3_ture)

# 计算每个RA的余弦相似度
cos_sim1 = cosine_similarity(RA1, RA1_true)
cos_sim2 = cosine_similarity(RA2, RA2_true)
cos_sim3 = cosine_similarity(RA3, RA3_true)

# 计算平均余弦相似度
all_cos_sim = [cos_sim1, cos_sim2, cos_sim3]
average_cos_sim = np.mean(all_cos_sim)

print(average_cos_sim)

0.4166666666666667


In [33]:
# 消融实验1: Proposed LLM vs LLM w/o Adapter
# 消融实验2: Proposed LLM vs LLM w/o Meta-Adapter
# 消融实验3: Proposed LLM vs MLP
from utils_new import mae, cosine_similarity, accuracy
import numpy as np

# Room 3-1: 选择的5个测试样本的APS要有差异性，因此选了前5个trace的第一个sample，对于RA选择的5个sample也一样规则，SINR为了方便选的就是第一个trace的前5个samples
RA1 = [[0.0, 1.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 0.0, 0.0, 1.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 0.0, 0.0, 0.0, 1.0]]
RA1_ture = '[[0.0 0.0 0.0 0.0 0.37 0.0] [0.13 0.0 0.0 0.0 0.0 0.0] [0.0 0.0 0.5 0.0 0.0 0.0] [0.0 0.0 0.0 0.0 0.0 0.26] [0.0 0.0 0.0 0.5 0.0 0.0] [0.13 0.0 0.0 0.0 0.0 0.0] [0.0 0.0 0.0 0.0 0.12 0.0] [0.0 0.0 0.0 0.0 0.37 0.0] [0.0 0.0 0.0 0.0 0.0 0.54] [0.0 0.0 0.5 0.0 0.0 0.0] [0.0 0.0 0.0 0.5 0.0 0.0] [0.09 0.0 0.0 0.0 0.0 0.0] [0.0 1.0 0.0 0.0 0.0 0.0] [0.14 0.0 0.0 0.0 0.0 0.0] [0.0 0.0 0.0 0.0 0.0 0.19] [0.06 0.0 0.0 0.0 0.0 0.0] [0.0 0.0 0.0 0.0 0.14 0.0] [0.19 0.0 0.0 0.0 0.0 0.0] [0.11 0.0 0.0 0.0 0.0 0.0] [0.15 0.0 0.0 0.0 0.0 0.0]]'
RA2 = [[0.0, 1.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 0.0, 0.0, 1.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 0.0, 0.0, 0.0, 1.0]]
RA2_ture = '[[0.0 0.0 0.0 0.0 0.35 0.0] [0.12 0.0 0.0 0.0 0.0 0.0] [0.0 0.0 0.0 0.0 0.0 0.5] [0.18 0.0 0.0 0.0 0.0 0.0] [0.0 0.0 0.0 0.0 0.0 0.5] [0.11 0.0 0.0 0.0 0.0 0.0] [0.07 0.0 0.0 0.0 0.0 0.0] [0.0 0.0 0.0 0.0 0.35 0.0] [0.0 0.0 0.0 0.5 0.0 0.0] [0.0 0.0 0.36 0.0 0.0 0.0] [0.0 0.68 0.0 0.0 0.0 0.0] [0.1 0.0 0.0 0.0 0.0 0.0] [0.0 0.0 0.36 0.0 0.0 0.0] [0.0 0.0 0.0 0.0 0.3 0.0] [0.15 0.0 0.0 0.0 0.0 0.0] [0.04 0.0 0.0 0.0 0.0 0.0] [0.08 0.0 0.0 0.0 0.0 0.0] [0.0 0.0 0.0 0.5 0.0 0.0] [0.15 0.0 0.0 0.0 0.0 0.0] [0.0 0.0 0.29 0.0 0.0 0.0]]'
RA3 = [[0.0, 0.0, 1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 0.0, 0.0, 0.0, 1.0], [1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 0.0, 0.0, 0.0, 1.0]]
RA3_ture = '[[0.0 0.0 0.0 0.0 0.0 0.3] [0.0 0.0 0.54 0.0 0.0 0.0] [0.0 0.0 0.0 1.0 0.0 0.0] [0.14 0.0 0.0 0.0 0.0 0.0] [0.07 0.0 0.0 0.0 0.0 0.0] [0.0 0.0 0.46 0.0 0.0 0.0] [0.0 0.0 0.0 0.0 0.24 0.0] [0.37 0.0 0.0 0.0 0.0 0.0] [0.05 0.0 0.0 0.0 0.0 0.0] [0.0 0.5 0.0 0.0 0.0 0.0] [0.0 0.0 0.0 0.0 0.0 0.43] [0.34 0.0 0.0 0.0 0.0 0.0] [0.0 0.5 0.0 0.0 0.0 0.0] [0.04 0.0 0.0 0.0 0.0 0.0] [0.0 0.0 0.0 0.0 0.76 0.0]]'


def cosine_similarity(list1, list2):
    vec1 = np.array(list1)
    vec2 = np.array(list2)
    if vec1.ndim == 1 and vec2.ndim == 1:
        dot_product = np.dot(vec1, vec2)
        norm1 = np.linalg.norm(vec1)
        norm2 = np.linalg.norm(vec2)
        if norm1 == 0 or norm2 == 0:
            return 0
        return dot_product / (norm1 * norm2)
    elif vec1.ndim == 2 and vec2.ndim == 2 and vec1.shape == vec2.shape:
        dot_products = np.sum(vec1 * vec2, axis=1)
        norms1 = np.linalg.norm(vec1, axis=1)
        norms2 = np.linalg.norm(vec2, axis=1)
        norms_product = norms1 * norms2
        norms_product[norms_product == 0] = 1
        similarities = dot_products / norms_product
        return np.mean(similarities)
    else:
        raise ValueError("输入必须是相同形状的一维向量或二维数组。")

# 解析字符串格式的真实值
def parse_ra_target(target_str):
    # 移除空格和方括号
    target_str = target_str.strip()
    # 使用正则表达式提取所有数字
    numbers = re.findall(r'[-+]?\d*\.?\d+', target_str)
    # 转换为浮点数
    numbers = [float(num) for num in numbers]
    # 根据数据推断形状
    if target_str.count('[') == 2:  # 2D数组
        # 计算每行有多少列（假设是5列）
        num_elements = len(numbers)
        num_rows = num_elements // 6
        return np.array(numbers).reshape(num_rows, 6)
    else:
        return np.array(numbers).reshape(-1, 6)

# 解析所有真实值
RA1_true = parse_ra_target(RA1_ture)
RA2_true = parse_ra_target(RA2_ture)
RA3_true = parse_ra_target(RA3_ture)

# 计算每个RA的余弦相似度
cos_sim1 = cosine_similarity(RA1, RA1_true)
cos_sim2 = cosine_similarity(RA2, RA2_true)
cos_sim3 = cosine_similarity(RA3, RA3_true)

# 计算平均余弦相似度
all_cos_sim = [cos_sim1, cos_sim2, cos_sim3]
average_cos_sim = np.mean(all_cos_sim)

print(average_cos_sim)

0.20555555555555557


In [ ]:
from transformers import AutoTokenizer  # 改用AutoTokenizer自动匹配Tokenizer类型

# 加载Llama-3.1-8B-Instruct的分词器（自动适配Fast版本）
tokenizer = AutoTokenizer.from_pretrained(
    "/data/LLM_indoor/LLaMA-Factory-main/models/Llama-3.1-8B-Instruct",
    legacy=False 
)
# 要计算的文本
text = '[56.83, 26.10, 24.23, 33.76, 30.03]'
# 计算Token数（encode返回Token ID列表，len即数量）
token_count = len(tokenizer.encode(text))
print(f"文本：{text}")
print(f"Token数:{token_count}")  # Llama-3.1下输出：31（带空格版）/26（无空格版）

文本：[56.83, 26.10, 24.23, 33.76, 30.03]
Token数：26


In [7]:
# 画图,计算不同Room下的不同UE数目的Sum-rate
import csv
from collections import defaultdict

def calculate_avg_thr(file_path):
    tar_thr_dict = defaultdict(list)
    pre_thr_dict = defaultdict(list)
    
    with open(file_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            ue_num = int(row['UE_num'])
            target_thr = float(row['Target_Thr'])
            pre_thr = float(row['Pred_Thr'])
            tar_thr_dict[ue_num].append(target_thr)
            pre_thr_dict[ue_num].append(pre_thr)
    
    ue_list = sorted(tar_thr_dict.keys())
    tar_thr_list = [round(sum(tar_thr_dict[ue]) / len(tar_thr_dict[ue]), 3) for ue in ue_list]
    pre_thr_list = [round(sum(pre_thr_dict[ue]) / len(pre_thr_dict[ue]), 3) for ue in ue_list]
    
    return ue_list, tar_thr_list, pre_thr_list

data_path = '/data/LLM_indoor/LLaMA-Factory-main/LLM-train/Multitask/results/Dataset3_10k_RArevised-20251228-0450PM/Dataset3_room6-1_sharegpt_5k_RArevised/result_throuput.csv'
ue_list, tar_thr_list, pre_thr_list = calculate_avg_thr(data_path)
print(ue_list)
print(tar_thr_list)
print(pre_thr_list)

[10, 20, 30, 40]
[695.009, 1140.386, 1443.417, 1392.047]
[666.884, 1078.38, 1331.887, 1183.262]
